In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [20]:
import os
# ⬇️ Параметры подключения к PostgreSQL
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name = "public.shops"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

shops_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("user", db_user)
    .option("password", db_password)
    .option("dbtable", table_name)
    .option("fetchsize", 1000)
    .option("driver", "org.postgresql.Driver")
    .load() 
)

shops_df.show()
shops_df.printSchema()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+

root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)



In [24]:
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 

shop_timezone_df = (
    spark.read 
    .format("jdbc") 
    .option("url", jdbc_url) 
    .option("user", db_user) 
    .option("password", db_password) 
    .option("dbtable", table_name) 
    .option("fetchsize", 1000)
    .option("driver", "org.postgresql.Driver")
    .load()
)

shop_timezone_df.show()


+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [18]:
from pyspark.sql import functions as F
from pyspark.sql import Window

shop_timezone_df = (
    shop_timezone_df
    .select(F.col('plant').cast('int'), 'time_zone' )
)
shops_df = (
    shops_df
    .select(F.col('st_id').cast('int'), 'shop_name')
)

joined = shops_df.join(shop_timezone_df, shops_df.st_id == shop_timezone_df.plant, "inner")
#joined.show()
w = Window.partitionBy(F.col('plant')).orderBy(F.col('plant'))

temp = (
    joined
    .withColumn('rnk', F.row_number().over(w))
)
#temp.show()
result = (
    temp
    .withColumn('tz_code', F.when(
        (F.col('time_zone') == '') | (F.col('time_zone').isNull()), F.lit(3))
        .otherwise(F.substring(F.col('time_zone'), 4, 100)).cast('int'))
    .where(F.col('rnk') == 1)
    .select('st_id', 'shop_name', 'tz_code')
)
result.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [25]:
spark.stop()